In [15]:
import os
import sys
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import json
from datasets import load_dataset

from pyperf.harness.utils import natural_sort_key
from pyperf.constants import SUBMISSIONS_DIR

In [17]:
dataset = load_dataset("manishs/pyperf-extended", split="test")
pyperf_df = dataset.to_pandas()

README.md:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.44M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/122 [00:00<?, ? examples/s]

In [2]:
# helpers
plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)


def save_plot(fig, fname):
    fpath = os.path.join(plots_dir, fname)
    fig.savefig(fpath, dpi=300, bbox_inches="tight")
    print(f"Saved plot to {fpath}")
    plt.close(fig)


def get_traces(pattern: str):
    traces = list(SUBMISSIONS_DIR.glob(pattern))
    if traces == []:
        print(f"Could not find any traces matching pattern: {pattern}")
    traces = sorted([str(t) for t in traces], key=natural_sort_key)
    return [Path(t) for t in traces]


def load_json(fname):
    with open(fname, "r") as f:
        return json.load(f)

In [3]:
HOME_DIR = Path(os.path.expanduser("~"))
EVAL_REPORTS_DIR = HOME_DIR / "pyperf/reports/beat_k_reports"
eval_report = EVAL_REPORTS_DIR / "claude.beat@10.test.report.json"
traces = get_traces(
    "manishs__pyperf-test/CodeActAgent/claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0.25.0-no-hint-run_*/output.jsonl"
)
submissions = get_traces(
    "manishs__pyperf-test/CodeActAgent/claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0.25.0-no-hint-run_*/output.pyperf.jsonl"
)

In [4]:
report = load_json(eval_report)

# %% Load traces and create a mapping from instance_id to trace
import pandas as pd
from tqdm import tqdm


def load_traces_to_df(trace_files):
    traces_data = []

    for trace_file in tqdm(trace_files, desc="Loading trace files"):
        run_id = trace_file.parent.name  # Extract run ID from path

        with open(trace_file, "r") as f:
            for line in f:
                try:
                    trace = json.loads(line)
                    if "instance_id" in trace:
                        trace["run_id"] = run_id
                        traces_data.append(trace)
                except json.JSONDecodeError:
                    continue

    return pd.DataFrame(traces_data)


# Load all traces into a dataframe
traces_df = load_traces_to_df(traces)

# Print info about the dataframe
print(f"Loaded {len(traces_df)} trace entries")
print(f"Unique instance IDs: {traces_df['instance_id'].nunique()}")
print(f"Unique run IDs: {traces_df['run_id'].nunique()}")
traces_df.head()

Loading trace files: 100%|██████████| 10/10 [00:06<00:00,  1.45it/s]


Loaded 1220 trace entries
Unique instance IDs: 122
Unique run IDs: 10


,instance_id,test_result,instruction,metadata,history,metrics,error,instance,run_id
0,numpy__numpy-3ea71da,{'git_patch': ''},<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-17T21:57:48.4...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",None,"{'instance_id': 'numpy__numpy-3ea71da', 'repo'...",claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....
1,numpy__numpy-794f474,{'git_patch': ''},<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-17T21:57:46.7...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",None,"{'instance_id': 'numpy__numpy-794f474', 'repo'...",claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....
2,numpy__numpy-893db31,{'git_patch': 'diff --git a/numpy/_core/_strin...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-17T21:58:23.7...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",None,"{'instance_id': 'numpy__numpy-893db31', 'repo'...",claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....
3,numpy__numpy-f69f273,{'git_patch': 'diff --git a/numpy/core/setup.p...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-17T21:57:53.1...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,"{'instance_id': 'numpy__numpy-f69f273', 'repo'...",claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....
4,numpy__numpy-83c780d,{'git_patch': 'diff --git a/numpy/_core/defcha...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-17T21:57:48.9...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,"{'instance_id': 'numpy__numpy-83c780d', 'repo'...",claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....


In [5]:
opt_stats = report["opt_stats"]
opt_stats_data = []

for instance_id, stats in opt_stats.items():
    run_id = stats["report_file"].split("/")[-1].rstrip("test.report.json")

    with open(HOME_DIR / "pyperf" / stats["report_file"], "r") as f:
        run_report = json.load(f)
        good_set = run_report["instance_sets"]["improved_commit_ids"]

    stats_dict = {
        "instance_id": instance_id,
        "run_id": run_id,
        "is_good": instance_id in good_set,
        **stats,  # Unpack all stats
    }
    opt_stats_data.append(stats_dict)

opt_stats_df = pd.DataFrame(opt_stats_data)
print(f"Loaded {len(opt_stats_df)} entries from opt_stats")
opt_stats_df.head()

Loaded 75 entries from opt_stats


,instance_id,run_id,is_good,opt_perc_base,opt_perc_commit,opt_perc_main,speedup_base,speedup_commit,speedup_main,geomean_speedup_base,...,base_mean,base_std,patch_mean,patch_std,commit_mean,commit_std,main_mean,main_std,per_test_means,report_file
0,huggingface__datasets-c5464b3,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,76.588621,NaN,NaN,4.271427,NaN,NaN,4.271427,...,3.218526,0.000000,0.753501,0.000000,0.982287,0.000000,NaN,NaN,"{'base': [3.2185263999999996], 'patch': [0.753...",reports/claude-3-5-sonnet-v2-20241022_maxiter_...
1,numpy__numpy-f69f273,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,42.994981,NaN,22.074507,1.754232,NaN,1.283277,1.841431,...,0.092337,0.066569,0.052637,0.045215,0.064137,0.045121,0.067548,0.050887,"{'base': [0.16659419999999997, 0.0724102000000...",reports/claude-3-5-sonnet-v2-20241022_maxiter_...
2,pandas-dev__pandas-15fd7d7,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,30.053476,NaN,NaN,1.429664,NaN,NaN,1.418701,...,0.000093,0.000017,0.000065,0.000003,0.000040,0.000011,0.000071,0.000012,"{'base': [8.16e-05, 0.0001054], 'patch': [6.34...",reports/claude-3-5-sonnet-v2-20241022_maxiter_...
3,pandas-dev__pandas-191557d,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,99.541136,NaN,91.362402,217.929442,NaN,11.577293,217.929442,...,0.135901,0.000000,0.000624,0.000000,0.000615,0.000000,0.007220,0.000000,"{'base': [0.1359008], 'patch': [0.0006236], 'c...",reports/claude-3-5-sonnet-v2-20241022_maxiter_...
4,pandas-dev__pandas-2278923,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,True,94.193894,80.524614,61.969360,17.223247,5.134686,2.629459,17.108927,...,0.003734,0.000544,0.000217,0.000004,0.001113,0.000010,0.000570,0.000028,"{'base': [0.0033568, 0.0043582000000000004, 0....",reports/claude-3-5-sonnet-v2-20241022_maxiter_...


In [6]:
traces_df_last = traces_df.sort_values(["instance_id", "run_id"])

# Merge with opt_stats
merged_df = pd.merge(
    traces_df_last,
    opt_stats_df,
    on=["instance_id", "run_id"],
    how="outer",
    suffixes=("_trace", "_stats"),
)

merged_df["is_good"] = merged_df["is_good"].fillna(False)

# Check merge results
print(f"Merged dataframe has {len(merged_df)} rows")
print(
    f"Number of instances with both trace and stats: {merged_df.dropna(subset=['instance_id', 'run_id']).shape[0]}"
)

# Display the merged dataframe
merged_df.head()

Merged dataframe has 1220 rows
Number of instances with both trace and stats: 1220


/tmp/ipykernel_4144856/3821476851.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df["is_good"] = merged_df["is_good"].fillna(False)


,instance_id,test_result,instruction,metadata,history,metrics,error,instance,run_id,is_good,...,base_mean,base_std,patch_mean,patch_std,commit_mean,commit_std,main_mean,main_std,per_test_means,report_file
0,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/confi...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-21T05:28:12.4...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,{'instance_id': 'huggingface__datasets-2878019...,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,huggingface__datasets-2878019,{},None,None,None,None,Timeout after 7200 seconds,None,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/arrow...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-21T06:15:21.0...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,{'instance_id': 'huggingface__datasets-2878019...,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/arrow...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-21T06:31:32.6...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,{'instance_id': 'huggingface__datasets-2878019...,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-21T06:44:56.3...","{'accumulated_cost': 0.0, 'costs': [], 'respon...",RuntimeError: Agent reached maximum iteration ...,{'instance_id': 'huggingface__datasets-2878019...,claude-3-5-sonnet-v2-20241022_maxiter_50_N_v0....,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Qualitative Analysis of trajectories by LLM

In [7]:
from r2e.llms.completions import LLMCompletions
import os
import re
import tiktoken
from r2e.llms.llm_args import LLMArgs

tokenizer = tiktoken.encoding_for_model("gpt-4")

In [8]:
def count_tokens(context: str):
    return len(tokenizer.encode(context, disallowed_special=()))

In [9]:
def history_to_string(history_list, max_tokens=500):
    """
    Converts history list into a string format with each item structured as:

    STEP ID: X
    SOURCE: Y
    CONTENT: Z

    Parameters:
    - history_list: List of history dictionaries
    - max_tokens: Maximum number of tokens for long text fields

    Returns:
    - String representation of the history
    """
    # Handle None or empty history
    if history_list is None or len(history_list) == 0:
        return "NO HISTORY AVAILABLE"

    result = []

    for item in history_list:
        # Start building the string representation for this item
        step_str = [f"STEP ID: {item.get('id', 'N/A')}"]
        step_str.append(f"SOURCE: {item.get('source', 'N/A')}")

        # Handle content based on the source
        if item.get("source") == "user":
            # For user messages, use the 'message' field or 'args.content' field
            content = item.get("message", "")
            if (
                not content
                and "args" in item
                and isinstance(item["args"], dict)
                and "content" in item["args"]
            ):
                content = item["args"]["content"]

            # If content is too long, truncate it
            if content and count_tokens(content) > max_tokens:
                # Approximate truncation
                approx_chars = int(
                    max_tokens * 3.5
                )  # Rough estimate: 1 token ≈ 3-4 chars
                content = content[:approx_chars] + "...(truncated)..."

            step_str.append(f"CONTENT: {content}")

        elif item.get("source") == "agent":
            # Check if this is an observation
            if "observation" in item:
                content = f"OBSERVATION: {item.get('observation', '')}"

                # If there's additional content in the 'content' field, add it
                if "content" in item and item["content"]:
                    observation_content = item["content"]
                    if count_tokens(observation_content) > max_tokens:
                        approx_chars = int(max_tokens * 3.5)
                        observation_content = (
                            observation_content[:approx_chars] + "...(truncated)..."
                        )
                    content += f"\n{observation_content}"

                step_str.append(f"CONTENT: {content}")
            else:
                # Try to get the model response
                try:
                    if (
                        "tool_call_metadata" in item
                        and "model_response" in item["tool_call_metadata"]
                    ):
                        model_response = item["tool_call_metadata"]["model_response"]
                        choices = model_response.get("choices", [])
                        if choices and len(choices) > 0:
                            content = choices[0].get("message", {}).get("content", "")

                            # If content is too long, truncate it
                            if content and count_tokens(content) > max_tokens:
                                approx_chars = int(max_tokens * 3.5)
                                content = content[:approx_chars] + "...(truncated)..."

                            step_str.append(f"CONTENT: {content}")
                        else:
                            # Fallback to message
                            content = item.get("message", "")
                            step_str.append(f"CONTENT: {content}")
                    else:
                        # Fallback to message
                        content = item.get("message", "")
                        step_str.append(f"CONTENT: {content}")
                except Exception as e:
                    # If any error occurs, use the message field as fallback
                    content = item.get("message", f"Error parsing content: {str(e)}")
                    step_str.append(f"CONTENT: {content}")
        else:
            # For other sources, just use the message field
            content = item.get("message", "")
            step_str.append(f"CONTENT: {content}")

        # Add this step's string to the result
        result.append("\n".join(step_str))

    # Join all steps with double newlines for separation
    return "\n\n".join(result)

In [10]:
from tqdm.notebook import tqdm

tqdm.pandas()

MAX_CONTENT_TOKENS = 500
merged_df["compact_history"] = merged_df["history"].progress_apply(
    lambda x: (
        history_to_string(x, MAX_CONTENT_TOKENS)
        if x is not None
        else "NO HISTORY AVAILABLE"
    )
)

  0%|          | 0/1220 [00:00<?, ?it/s]

In [25]:
analysis_prompt = """Here's an agent tasked at optimizing a codebase to improve a performance test. 
Your task is to analyze the trajectory and write a nice report!

GUIDELINES YOU MUST FOLLOW:
1. Keep it concise, short, and well classified. Use points to write it, with headings indicating the key phrase of the point. 
    - e.g., "Tunnel Vision:...", "Premature Optimization:...", etc. But be original with your headings but not cheeky.
2. MOST IMPORTANT: Analyze and study and capture the behavioural patterns and deep insights into why/why not the agent was successful. Below are some examples, but you should go above and beyond:
    - e.g., misdiagnosis of opportunities
    - e.g., wrong choice of algorithm or decision
    - e.g., lazy optimization / premature optimization
    - Ignore basic things like the agent following the steps I provided, exploring and reading codebase, iterative testing, etc. THESE ARE NOT IMPORTANT.
    - NOTE: DO NOT JUST STICK TO THESE EXAMPLES AND JUST REPEAT THEM. GO DEEPER!
3. Sprinkle in some deep technical challenges that either the agent solved or faced.
4. AGAIN DO NOT COVER BASIC STUFF like reading and exploring codebase.
5. Don't bother suggesting fixes or improvements. Just analyze the trajectory and write a report.
6. PLEASE FOCUS ON THE BEHAVIOUT OF THE AGENT MAINLY AND HOW IT IMPACTED THE SUCCESS/FAILURE OF THE OPTIMIZATION
7. Use the STEP ID to refer to specific steps you think were critical points in the trajectory that affected the outcome.
8. Focus on the behaviours that truly affected the optimization status (as indicated below).

Next you will see: 
- compact history of the agent's trajectory
- a human optimization diff: diff provided by a developer that gets good speedup on the same task. use to compare the solution of the agent with the human optimization.
- status (successful optmisation that beats human diff or not depending on how much speedup was achieved)

## Trajectory:
{trajectory}

## Human Optimization Diff:
{human_diff}

## Successful Optimization Status: {status}

Find the gems! Keep it to 5-6 points if possible. 
"""


def prompt_o3_mini(status, human_diff, trajectory):
    return analysis_prompt.format(
        status=status, human_diff=human_diff, trajectory=trajectory
    )

In [26]:
temp_df = merged_df

payloads = []
for _, row in temp_df.iterrows():
    diff = pyperf_df[pyperf_df["instance_id"] == row["instance_id"]]["gt_diff"].values[
        0
    ]
    prompt = prompt_o3_mini(row["is_good"], diff, row["compact_history"])
    payloads.append([{"role": "user", "content": prompt}])

outputs = LLMCompletions.get_llm_completions(
    LLMArgs(
        model_name="o3-mini",
        multiprocess=30,
        max_tokens=24000,
        use_cache=True,
        cache_batch_size=100,
    ),
    payloads,
)

analyses = [item for sublist in outputs for item in sublist]

100%|██████████| 20/20 [00:14<00:00,  1.38it/s, exc=0, p_exp=0, succ=20, timeouts=0]


In [27]:
# add the analysis as new column to the dataframe
temp_df["analysis"] = analyses

# keep only the relevant columns: instance_id, run_id, and analysis
temp_df = temp_df[["instance_id", "run_id", "analysis"]]
temp_df.head()

# save the dataframe to a CSV file
temp_df.to_csv("trajectory_analysis.csv", index=False)
print("Saved merged dataframe to trajectory_analysis.csv")

Saved merged dataframe to trajectory_analysis.csv
